# 01 — Data Exploration

**ML Ensemble Benchmarking Framework**

This notebook explores the raw dataset before any preprocessing: shape, dtypes, missing values, target distribution, and basic feature correlations. It uses the same `src/data/load_data.py` loader as the rest of the pipeline, so this exploration reflects exactly what the benchmarking code will see.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_raw_data
from src.config import load_config

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## Load Configuration and Data

In [ ]:
config = load_config()

df = load_raw_data(
    path=config.get("data.raw_path"),
    target_column=config.get("data.target_column", "target"),
    n_samples=config.get("data.n_samples", 50000),
    n_features=config.get("data.n_features", 20),
)

print(f"Dataset shape: {df.shape}")
df.head()

## Data Types and Missing Values

In [ ]:
print("Data types:")
print(df.dtypes.value_counts())
print()

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(missing if len(missing) else "None")

## Target Distribution

Confirms the class imbalance that motivates the SMOTE / class-weight handling later in the pipeline.

In [ ]:
target_col = config.get("data.target_column", "target")
counts = df[target_col].value_counts()
ratios = df[target_col].value_counts(normalize=True)

print(counts)
print()
print((ratios * 100).round(2).astype(str) + "%")

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x=df[target_col], ax=ax)
ax.set_title("Target Class Distribution")
ax.set_xlabel("Class")
plt.show()

## Numeric Feature Distributions

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.drop(target_col)

fig, axes = plt.subplots(4, 5, figsize=(18, 12))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col].dropna(), ax=ax, kde=True, bins=30)
    ax.set_title(col, fontsize=9)

for ax in axes.flatten()[len(numeric_cols):]:
    ax.set_visible(False)

fig.suptitle("Numeric Feature Distributions", y=1.02)
fig.tight_layout()
plt.show()

## Categorical Feature Breakdown

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()

for col in categorical_cols:
    print(df[col].value_counts())
    print()

## Correlation Heatmap (Numeric Features)

In [ ]:
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, square=True, cbar_kws={"shrink": 0.7})
ax.set_title("Feature Correlation Matrix")
plt.show()

## Key Observations

- The target is imbalanced (~15% positive class), which motivates the SMOTE / class-weight strategies used later in `03_benchmark_rf.ipynb` onward.
- A small fraction of `feature_0` is missing — handled by median imputation in `src/data/preprocess.py`.
- Two categorical columns (`category_region`, `category_channel`) require one-hot encoding before modeling.
- No numeric feature pair shows extreme collinearity (|corr| > 0.9), so multicollinearity is not an immediate concern ahead of feature engineering.

**Next notebook:** [`02_feature_engineering.ipynb`](02_feature_engineering.ipynb) applies preprocessing and the feature engineering pipeline, and quantifies the resulting improvement in model input quality.